# The athletic gap in the trenches: Combine tracking vs. NFL pass-protection reps

**NFL Big Data Bowl 2027 - reproducibility notebook.** This notebook rebuilds, from the competition CSVs, every number and figure of the writeup that can be rebuilt without a Bayesian sampler, and it loads the rest from the frozen confirmatory run that is committed with the code.

| Step | What it does | What it must reproduce |
|---|---|---|
| 1 | CSV to Parquet (`bdb2027.convert`) | Row counts of the nine tables (463,189 Combine tracking rows, 314,197 `player_play` rows, ...) |
| 2 | Engaged pass-protection reps (`bdb2027.engaged_reps`) | 29,269 blocker-rusher pairs, 8,271 engaged reps, 107 blockers, 129 rushers |
| 3 | Combine trait qualification and frozen composites (`bdb2027.traits`) | OL composite = {O3 mirror reversal time}, DL composite = {D1 rush-drill peak speed}; r = 0.704 and 0.639 |
| 4 | Pre-registered analysis sets | P: 7,214 reps / 69 blockers / 97 rushers / 1,201 dyads; placebo set PL: 17,795 reps |
| 5 | Positive control PC1 and the GLM mirror of M1, M2, S1 | PC1 pick -0.204, dropback 0.577; M1 gap -0.154 (kappa CI [-0.308, 0.000]); M2 gap -0.138; S1 c_B +0.143, c_R -0.064 |
| 6 | Frozen Bayesian results (loaded, not refitted) | The verdict label and convergence table R2 of `reports/confirmatory_results.json` |
| 7 | Figures F1-F6 | The PNGs used in the writeup |
| 8 | Reproducibility report | Package versions, input hashes against the pre-registration manifest, a PASS line per assertion |

**What is not refitted here.** The Bayesian models (PyMC with nutpie) took 64 minutes on 32 CPUs and are not part of this notebook; their results are read from the committed `reports/confirmatory_results.json`. Nothing in this notebook needs `pymc`, `nutpie` or `numba`.

**Order of the steps.** The trait-coverage gate of step 3 is defined against the engaged-rep population, so the engaged-rep dataset (step 2) is built first.

**Honest framing.** The pre-registration, the frozen script and its hash were fixed before any outcome was opened (tag `prereg-frozen`). The sealed outcome file is built in step 2 but only opened in steps 5 and 6, after the confirmatory run has been done and frozen; step 5 is therefore a reproduction of a finished analysis, not a new test. Every assertion below fails loudly (the notebook stops) if a number does not match.

In [ ]:
# ---- configuration -------------------------------------------------------------------------------
# On Kaggle the repository code (src/bdb2027, scripts/, reports/) is fetched with `git clone`; this needs
# "Internet" switched ON in the notebook settings. Alternative without internet: attach a Kaggle Dataset that
# contains a copy of the repository (any /kaggle/input/<name>/ with src/bdb2027 inside) and it is used instead.
REPO_URL = "https://github.com/Jopared/bdb2027-trench-matchup.git"   # public repository of the entry
REPO_REF = None                                  # optional branch or tag to check out, e.g. "prereg-frozen"

import contextlib, glob, hashlib, importlib, importlib.metadata as imd, io, json, os, re, runpy, shutil, subprocess, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _ensure_repo():
    """Make `import bdb2027` work: already installed, else a checkout next to this notebook, else clone + pip install -e."""
    try:
        import bdb2027  # noqa: F401
        return
    except ImportError:
        pass
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:                       # local checkout, package not installed
        if (base / "src" / "bdb2027").is_dir():
            sys.path.insert(0, str(base / "src"))
            return
    dest = Path("/kaggle/working/bdb2027-repo")
    if not dest.exists():
        found = glob.glob("/kaggle/input/*/src/bdb2027") + glob.glob("/kaggle/input/*/*/src/bdb2027")
        if found:                                            # repository attached as a dataset: copy (inputs are read-only)
            shutil.copytree(Path(found[0]).parents[1], dest)
        else:                                                # needs Internet ON
            subprocess.run(["git", "clone", "--depth", "1"] + (["--branch", REPO_REF] if REPO_REF else []) + [REPO_URL, str(dest)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(dest), "--no-deps"], check=True)
    sys.path.insert(0, str(dest / "src"))                    # .pth files are not re-read by a running interpreter
    importlib.invalidate_caches()


_ensure_repo()

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import Image, Markdown, display

from bdb2027 import paths

REPO = Path(paths.REPO_ROOT)
SCRIPTS = REPO / "scripts"
assert (SCRIPTS / "confirmatory_analysis.py").exists() and (REPO / "reports" / "preregistration.md").exists(), \
    "repository checkout incomplete (scripts/ and reports/ are needed)"
OUT_DIR = Path(os.environ.get("BDB_OUT_DIR", (Path("/kaggle/working") if paths.ON_KAGGLE else REPO / "data") / "notebook_outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
REBUILD = os.environ.get("BDB_NB_REBUILD") == "1"        # rebuild even if the Parquet outputs already exist
LOCAL_REFERENCE = (REPO / "data" / "processed").resolve()  # the frozen local data products (their hashes are registered)
if REBUILD and Path(paths.PROCESSED_DIR).resolve() == LOCAL_REFERENCE:
    raise RuntimeError("BDB_NB_REBUILD=1 would overwrite the frozen data/processed files; set BDB_PROCESSED_DIR to another folder.")

CHECKS, TIMES = [], {}


def check(name, ok, detail=""):
    """Record an assertion, print a PASS/FAIL line, and stop the notebook on failure."""
    CHECKS.append((name, bool(ok), detail))
    print(("PASS  " if ok else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))
    assert ok, f"{name}: {detail}"


@contextlib.contextmanager
def timed(label):
    t = time.time()
    try:
        yield
    finally:
        TIMES[label] = TIMES.get(label, 0.0) + time.time() - t
        print(f"[{label}] {TIMES[label]:.1f} s")


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


print("on Kaggle:", paths.ON_KAGGLE)
print("repo root:", REPO)
print("raw CSVs :", paths.RAW_DIR)
print("processed:", paths.PROCESSED_DIR, "(reused if present; set BDB_NB_REBUILD=1 to rebuild)")
print("outputs  :", OUT_DIR)

## 1. CSV to Parquet

`bdb2027.convert` turns the nine competition CSVs into Parquet. Two traps from the host forum are handled there: the literal string `"NA"` in `player_play.lined_up_position` marks plays killed by a pre-snap penalty and is kept as a string, and `pass_result` is the empty string (not null) on non-dropback plays. Tables whose Parquet file already exists in the processed folder are reused, not reconverted (this is how the local run keeps the registered file hashes intact).

**Must reproduce:** 463,189 Combine tracking rows; 314,197 `player_play` rows; 3,665,443 / 7,827,944 / 11,160,903 game-tracking rows (2023 / 2024 / 2025); 510 players, 510 Combine results, 510 career rows; 1,002 games; 103,636 non-dropback rows with an empty `pass_result`; the literal `"NA"` alignment kept.

In [ ]:
from bdb2027 import convert

EXPECTED_ROWS = {"players": 510, "combine_results": 510, "combine_tracking": 463_189, "player_career_successes": 510,
                 "player_play": 314_197, "games": 1_002, "game_tracking_2023": 3_665_443,
                 "game_tracking_2024": 7_827_944, "game_tracking_2025": 11_160_903}
rows = []
with timed("1 conversion"):
    for t in paths.TABLES:
        dst = paths.parquet(t)
        t0 = time.time()
        if dst.exists() and not REBUILD:
            how = "reused"
        else:
            if not paths.raw_csv(t).exists():
                raise FileNotFoundError(f"{paths.raw_csv(t)} is missing; join the competition and download the data first")
            with contextlib.redirect_stdout(io.StringIO()):
                convert.convert(t)
            how = "converted"
        n = pl.scan_parquet(dst).select(pl.len()).collect().item()
        rows.append(dict(table=t, rows=n, expected=EXPECTED_ROWS[t], how=how, seconds=round(time.time() - t0, 1)))
display(pd.DataFrame(rows).style.format({"rows": "{:,}", "expected": "{:,}"}))
for r in rows:
    check(f"rows {r['table']} = {r['expected']:,}", r["rows"] == r["expected"], f"{r['rows']:,}")

pp = pl.scan_parquet(paths.parquet("player_play"))
n_empty = pp.filter(pl.col("pass_result") == "").select(pl.len()).collect().item()
n_na = pp.filter(pl.col("lined_up_position") == "NA").select(pl.len()).collect().item()
n_null_pr = pp.filter(pl.col("pass_result").is_null()).select(pl.len()).collect().item()
check("player_play: 103,636 rows with empty-string pass_result (non-dropbacks), none null", n_empty == 103_636 and n_null_pr == 0, f"{n_empty:,} empty, {n_null_pr} null")
check('player_play: literal "NA" lined_up_position kept as a string', n_na > 0, f"{n_na:,} rows")

## 2. Engaged pass-protection reps

A **pair** is a blocker and a pass rusher from the cohort who are on the field for the same dropback. A pair is **engaged** when both players are tracked within 1.5 yd of each other on at least 4 frames between 0.2 s after the snap and the earliest of snap + 3.0 s, the pass being thrown, or a sack. The builder (`bdb2027.engaged_reps`) writes three files: the open pairs table, the player baselines, and a *sealed* file that holds every outcome column; no model is fitted here. (`scripts/build_engaged_reps.py` runs the same three calls and then writes a report; the notebook reimplements only the three calls and the three writes.)

**Must reproduce** (`reports/dataset_build.md` section 2): 29,269 pairs (REG+POST), 8,271 engaged reps, 107 distinct engaged blockers, 129 distinct engaged rushers, 510 baseline rows.

In [ ]:
from bdb2027 import engaged_reps as er

ENG_FILES = [paths.parquet(n) for n in ("engaged_pairs", "player_baselines", "engaged_outcomes_SEALED")]
with timed("2 engaged reps"):
    if all(f.exists() for f in ENG_FILES) and not REBUILD:
        print("engaged-rep files exist: reused")
        diag = None
    else:
        pairs_full, diag = er.build_engaged_pairs(None)
        open_, sealed = er.split_outcomes(pairs_full)
        base = er.build_player_baselines()
        open_.write_parquet(paths.parquet("engaged_pairs"))
        base.write_parquet(paths.parquet("player_baselines"))
        sealed.write_parquet(paths.parquet("engaged_outcomes_SEALED"))
        del pairs_full, open_, sealed, base
        print("built from the tracking Parquet files:", diag)

eng_pairs = pl.read_parquet(paths.parquet("engaged_pairs"))
e0 = eng_pairs.filter(pl.col("engaged_primary"))
sealed_keys = pl.read_parquet(paths.parquet("engaged_outcomes_SEALED"), columns=er.KEYS)   # key columns only
summary = pd.DataFrame({"quantity": ["pairs (REG+POST)", "engaged reps, primary", "distinct engaged blockers", "distinct engaged rushers"],
                        "this notebook": [eng_pairs.height, e0.height, e0["blocker_nfl_id"].n_unique(), e0["rusher_nfl_id"].n_unique()],
                        "reports/dataset_build.md": [29_269, 8_271, 107, 129]})
display(summary.style.format({"this notebook": "{:,}", "reports/dataset_build.md": "{:,}"}))
for _, r in summary.iterrows():
    check(f"engaged-rep build: {r['quantity']} = {r['reports/dataset_build.md']:,}", r["this notebook"] == r["reports/dataset_build.md"])
check("engaged-rep build: baselines hold the 510 cohort players", pl.read_parquet(paths.parquet("player_baselines")).height == 510)
check("engaged-rep build: sealed outcome file has the same 29,269 unique keys",
      sealed_keys.height == eng_pairs.height and sealed_keys.unique().height == eng_pairs.height)

## 3. Combine trait qualification and the frozen composites

The pool of candidate drill traits (O1-O5 for offensive linemen, D1-D6 for defensive linemen) is fixed, and each trait must pass three mechanical gates computed from Combine data only: **reliability** (it repeats across a player's own reps), **redundancy** (it is not just the stopwatch or body size, |r| < 0.75), and **coverage** (at least 80% of the engaged-rep players have a value). The passing traits form the frozen side composites: the mean of their within-position, within-draft-year z-scores.

`scripts/trait_qualification.py` does this and also writes `reports/trait_qualification.md` and overwrites `combine_traits.parquet`; its `main()` is therefore **not** called. The notebook runs the same calls on `bdb2027.traits` (identical code path, bootstrap seed 2027), reproduces the sensitivity-composite and parquet-column steps verbatim, and then compares the result with the stored `combine_traits.parquet` (or writes it, when it does not exist yet).

**Must reproduce:** OL composite = {O3}, DL composite = {D1}; O3 odd/even r = 0.704, D1 cross-rep r = 0.639; engaged populations of 79 blockers and 97 rushers; O4 and D5 fail the coverage gate at 63/79 and 77/97.

In [ ]:
from bdb2027 import traits as T

with timed("3 trait qualification"):
    cr = T.load_combine_results()
    reps = {k: T.load_reps(v) for k, v in T.DRILLS.items()}
    POP = T.engaged_populations()
    R3, V3 = {}, {}
    for tid in T.TRAITS:
        R3[tid] = T.evaluate_trait(tid, reps, cr, 3, pop=POP)
        V3[tid] = T.gate_verdicts(R3[tid])
    R5_O3 = T.evaluate_trait("O3", reps, cr, 5, pop=POP)          # smoothing sensitivity of the only OL component
    comp, info = T.build_composites(R3, V3, cr)

    # --- sensitivity composites and extra columns, verbatim from scripts/trait_qualification.py ---
    def sens(tids_by_side):
        c = pd.Series(np.nan, index=comp.index); n = pd.Series(0, index=comp.index)
        for side, tids in tids_by_side.items():
            m = comp.combine_position == side
            Z = comp.loc[m, [f"{t}_z" for t in tids]]
            c[m] = Z.mean(axis=1, skipna=True); n[m] = Z.notna().sum(axis=1)
        return c.where(n > 0), n
    sensA = {s: [t for t in T.TRAITS if T.TRAITS[t].side == s and V3[t]["rel"] and V3[t]["red"]] for s in ("OL", "DL")}
    sensB = {s: sorted(set(v) | ({"O1"} if s == "OL" else set())) for s, v in sensA.items()}
    comp["composite_sensA"], comp["composite_sensA_n"] = sens(sensA)
    comp["composite_sensB"], comp["composite_sensB_n"] = sens(sensB)
    for t in T.TRAITS:
        comp[f"{t}_gate_reliability"] = V3[t]["rel"]
        comp[f"{t}_gate_redundancy"] = V3[t]["red"]
        comp[f"{t}_gate_coverage"] = V3[t]["cov"]
        comp.loc[comp.combine_position != T.TRAITS[t].side, [f"{t}_value", f"{t}_z", f"{t}_n_slots"]] = np.nan
    comp["composite_components"] = comp.combine_position.map(lambda s: ",".join(info[s]["passing"]))


def fmt(v, d=3):
    return "n/a" if v is None or not np.isfinite(v) else f"{v:.{d}f}"


gate = pd.DataFrame([dict(ID=tid, side=tr.side, trait=tr.short,
                          r_with_95CI=("n/a" if not np.isfinite(R3[tid]["r"]) else f"{R3[tid]['r']:.3f} [{fmt(R3[tid]['lo'])}, {fmt(R3[tid]['hi'])}]"),
                          n=R3[tid]["n"], max_abs_r_vs_stopwatch_size=fmt(R3[tid]["max_abs_r"]),
                          coverage=f"{R3[tid]['coverage_n']}/{R3[tid]['n_pop']} = {100 * R3[tid]['coverage']:.1f}%",
                          reliability="PASS" if V3[tid]["rel"] else "fail", redundancy="PASS" if V3[tid]["red"] else "fail",
                          coverage_gate="PASS" if V3[tid]["cov"] else "fail", overall="PASS" if V3[tid]["passed"] else "fail")
                     for tid, tr in T.TRAITS.items()])
display(Markdown("**Gate table (3-frame smoothing, registered)**"))
display(gate)
print("OL composite components:", info["OL"]["passing"], "| DL composite components:", info["DL"]["passing"])
print(f"O3 with 5-frame smoothing: r = {R5_O3['r']:.3f} (below the 0.60 bar; the registered verdict uses 3-frame smoothing and is reported as fragile)")

check("traits: OL composite = {O3}", info["OL"]["passing"] == ["O3"], str(info["OL"]["passing"]))
check("traits: DL composite = {D1}", info["DL"]["passing"] == ["D1"], str(info["DL"]["passing"]))
check("traits: O3 odd/even r = 0.704", f"{R3['O3']['r']:.3f}" == "0.704", fmt(R3["O3"]["r"]))
check("traits: D1 cross-rep r = 0.639", f"{R3['D1']['r']:.3f}" == "0.639", fmt(R3["D1"]["r"]))
check("traits: engaged populations 79 blockers / 97 rushers", (len(POP["OL"]), len(POP["DL"])) == (79, 97), f"{len(POP['OL'])} / {len(POP['DL'])}")
check("traits: O4 and D5 fail only the coverage gate (63/79, 77/97)",
      (R3["O4"]["coverage_n"], R3["O4"]["n_pop"], R3["D5"]["coverage_n"], R3["D5"]["n_pop"]) == (63, 79, 77, 97) and not V3["O4"]["cov"] and not V3["D5"]["cov"] and V3["O4"]["rel"] and V3["D5"]["rel"])
check("traits: O3 5-frame r = 0.570", f"{R5_O3['r']:.3f}" == "0.570", fmt(R5_O3["r"]))

TRAITS_PQ = paths.parquet("combine_traits")
if TRAITS_PQ.exists() and not REBUILD:
    stored = pd.read_parquet(TRAITS_PQ)
    try:
        pd.testing.assert_frame_equal(comp.reset_index(drop=True), stored.reset_index(drop=True), check_exact=False, rtol=1e-9, atol=1e-9, check_dtype=False)
        same = True
    except AssertionError as e:
        same, msg = False, str(e)[:300]
    check("traits: recomputed composites equal the stored combine_traits.parquet", same, "" if same else msg)
else:
    comp.to_parquet(TRAITS_PQ, index=False)
    print("combine_traits.parquet written:", TRAITS_PQ)
    check("traits: combine_traits.parquet written with OL and DL composites", pd.read_parquet(TRAITS_PQ).composite.notna().sum() > 150)

## 4. The pre-registered analysis sets

The frame builder of the frozen script (`scripts/confirmatory_analysis.py`) is imported **read-only**, the same way the post-freeze exploratory work imported it (`reports/specification_log.md`, common machinery): `AnalysisBase(mode="exploratory", out_root=None).prepare(with_k5=False)`. This builds the frame, the composites per rep, the transformation constants and the named sets from the Parquet files, and writes nothing. `main()` is never called, no `--mode confirmatory` run is started, and the module-level code has no side effects other than setting `PYTENSOR_FLAGS` and trying to switch stdout to UTF-8. The frozen script's own consistency check (`v0_structural`) then compares 18 set definitions and 19 descriptive facts of pre-registration section 2.3 with the data.

**Must reproduce:** set **P** (engaged, both composites): 7,214 reps, 69 blockers, 97 rushers, 1,201 dyads, 431 / 1,821 / 4,962 reps in 2023 / 2024 / 2025, 6,867 regular-season reps; placebo set **PL** (E0 pairs that are not engaged): 17,795 reps; gap SD 1.253.

In [ ]:
import importlib.util

with timed("4 analysis sets"):
    _spec = importlib.util.spec_from_file_location("confirmatory_analysis_readonly", SCRIPTS / "confirmatory_analysis.py")
    CA = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(CA)                 # module-level code only defines names; main() is not called
    ana = CA.AnalysisBase(mode="exploratory", out_root=None)
    with contextlib.redirect_stdout(io.StringIO()):
        ana.prepare(with_k5=False)
        set_rows, mismatches = ana.v0_structural()
display(ana.r1_table(set_rows).drop(columns=["Pressures", "Rate"]))
P, PL = CA.set_counts(ana.F, ana.S["P"]), CA.set_counts(ana.F, ana.S["PL"])
check("frozen script V0: all section 2.3 set counts and 19 descriptive facts match", mismatches == [], "; ".join(mismatches[:3]))
check("set P = 7,214 reps / 69 blockers / 97 rushers / 1,201 dyads", (P["reps"], P["blockers"], P["rushers"], P["dyads"]) == (7214, 69, 97, 1201), str((P["reps"], P["blockers"], P["rushers"], P["dyads"])))
check("set P seasons 431 / 1,821 / 4,962 and 6,867 regular-season reps", (P["s2023"], P["s2024"], P["s2025"], P["reg"]) == (431, 1821, 4962, 6867))
check("placebo set PL = 17,795 reps", PL["reps"] == 17795, f"{PL['reps']:,}")
check("gap SD (registered, set P) = 1.253", abs(ana.V["reg"].gap_sd - 1.253) < 6e-4, f"{ana.V['reg'].gap_sd:.4f}")

## 5. Positive control and the frequentist mirror of the confirmatory models

**Outcome join (post-unsealing).** The sealed outcomes are joined 1:1 to the pairs on the four keys with the frozen script's own `attach_outcomes`, which asserts that no row is lost or duplicated.

**PC1 (pre-registration V1).** Before any gap model, the pipeline has to reproduce the earlier validated result: on all 8,271 engaged reps, a logit of `pressure_allowed` on z(rusher pick - blocker pick), z(rusher weight - blocker weight) and dropback duration, standard errors clustered by blocker. A failure would mean the outcome file is not the validated one. **Must reproduce:** 787 pressures, pick coefficient -0.204 (+/- 0.003), dropback 0.577 (+/- 0.003).

**GLM mirror (pre-registration section 5.6).** The same fixed-effect design matrix as the Bayesian models (built by the frozen `make_design` on set P with P's constants), a binomial GLM, and two-way cluster-robust standard errors (blocker + rusher - dyad), t reference with 68 degrees of freedom; the interval used for decisions is the kappa-calibrated one (standard errors x 1.20). M1 controls size, stopwatch times and the NGS athleticism score; M2 adds the draft-pick difference; S1 splits the gap into its rusher and blocker composites. **Must reproduce (3 decimals, equal to the frozen run):** M1 gap -0.154, kappa CI [-0.308, 0.000]; M2 gap -0.138; S1 c_B +0.143 and c_R -0.064.

The negative gap (a larger rusher-minus-blocker athletic gap went with *less* pressure) is the frozen finding; section 6 gives its label and caveats.

In [ ]:
with timed("5 PC1 and GLM mirror"):
    sealed_cols = ["game_id", "play_id", "blocker_nfl_id", "rusher_nfl_id", "pressure_allowed", "peak_pressure_probability_allowed", "rusher_time_to_pressure"]
    Y = pl.read_parquet(paths.parquet("engaged_outcomes_SEALED"), columns=sealed_cols).to_pandas()
    ana.attach_outcomes(Y)                                    # 1:1 join, asserts no rows lost or duplicated
    FROZEN = json.loads((REPO / "reports" / "confirmatory_results.json").read_text(encoding="utf-8"))

    pc1 = ana.pc1()
    cols_M1 = CA.C_COLS + CA.Q_COLS + ["gap"]
    cols_M2 = CA.C_COLS + CA.Q_COLS + CA.K_COLS + ["gap"]
    cols_S1 = CA.C_COLS + CA.Q_COLS + ["c_R", "c_B"]
    fits = {}
    with contextlib.redirect_stdout(io.StringIO()):
        for nm, cols in (("M1", cols_M1), ("M2", cols_M2), ("M1-S1", cols_S1)):
            fits[nm] = ana.fit_model(nm, ver="reg", setname="P", cols=cols, bayes=False)     # GLM only, no sampler

display(Markdown("**PC1 (positive control)**"))
display(pd.DataFrame([dict(quantity="events (pressures)", reproduced=pc1["events"], frozen=FROZEN["pc1"]["events"], target=787),
                      dict(quantity="pick coefficient", reproduced=round(pc1["pick"], 4), frozen=round(FROZEN["pc1"]["pick"], 4), target="-0.204 +/- 0.003"),
                      dict(quantity="dropback coefficient", reproduced=round(pc1["drop"], 4), frozen=round(FROZEN["pc1"]["drop"], 4), target="0.577 +/- 0.003"),
                      dict(quantity="rows", reproduced=pc1["n"], frozen=FROZEN["pc1"]["n"], target=8271)]))
check("PC1: 8,271 reps, 787 pressures, pick -0.204, dropback 0.577", pc1["pass"] and pc1["n"] == 8271 and pc1["events"] == 787,
      f"pick {pc1['pick']:.4f}, dropback {pc1['drop']:.4f}")
check("PC1 equals the frozen run", abs(pc1["pick"] - FROZEN["pc1"]["pick"]) < 1e-6 and abs(pc1["drop"] - FROZEN["pc1"]["drop"]) < 1e-6)


def frozen_r5(model, estimand):
    """Frozen GLM cell of table R5: 'est [kappa CI] {nominal CI}'."""
    row = next(r for r in FROZEN["tables"]["R5"] if r["Model"] == model and r["Estimand"] == estimand)
    return [float(x) for x in re.findall(r"-?\d+\.\d+", row["GLM [95% CI]"])]


rows = []
def compare(label, fit, term, frozen, tol=5e-4):   # frozen = (est, kappa_lo, kappa_hi, nominal_lo, nominal_hi)
    g = fits[fit].glm.term(term)
    got = (g["est"], *g["ci_kappa"], *g["ci_nom"])
    d = max(abs(a - b) for a, b in zip(got, frozen))
    rows.append(dict(model=fit, term=term, estimate=round(got[0], 3), kappa_CI=f"[{got[1]:.3f}, {got[2]:.3f}]", nominal_CI=f"[{got[3]:.3f}, {got[4]:.3f}]",
                     frozen_estimate=round(frozen[0], 3), frozen_kappa_CI=f"[{frozen[1]:.3f}, {frozen[2]:.3f}]", max_abs_diff=f"{d:.5f}", n=g["n"]))
    check(f"GLM mirror {label}: estimate and intervals equal the frozen run to 3 decimals", d < tol, f"est {got[0]:+.3f}, kappa CI [{got[1]:.3f}, {got[2]:.3f}]")


fz1, fz2 = FROZEN["glm_gap"]["M1"], FROZEN["glm_gap"]["M2"]
compare("M1 gap", "M1", "gap", (fz1["est"], *fz1["ci_kappa"], *fz1["ci_nom"]))
compare("M2 gap", "M2", "gap", (fz2["est"], *fz2["ci_kappa"], *fz2["ci_nom"]))
compare("S1 c_B", "M1-S1", "c_B", tuple(frozen_r5("M1-S1", "c_B")[:5]), tol=1e-3)   # table R5 prints 3 decimals
compare("S1 c_R", "M1-S1", "c_R", tuple(frozen_r5("M1-S1", "c_R")[:5]), tol=1e-3)
display(Markdown("**GLM mirror (log-odds of pressure allowed; rows = 7,214 reps, 69 blockers, 97 rushers)**"))
display(pd.DataFrame(rows))
check("M1 gap = -0.154 with kappa CI [-0.308, 0.000]", f"{fits['M1'].glm.term('gap')['est']:.3f}" == "-0.154" and f"{fits['M1'].glm.term('gap')['ci_kappa'][0]:.3f}" == "-0.308"
      and abs(fits["M1"].glm.term("gap")["ci_kappa"][1]) < 5e-4)
check("M2 gap = -0.138", f"{fits['M2'].glm.term('gap')['est']:.3f}" == "-0.138")
check("S1: c_B = +0.143, c_R = -0.064", f"{fits['M1-S1'].glm.term('c_B')['est']:+.3f}" == "+0.143" and f"{fits['M1-S1'].glm.term('c_R')['est']:+.3f}" == "-0.064")

## 6. The frozen Bayesian results (loaded, not refitted)

The headline inference of the writeup comes from the pre-registered Bayesian models (multilevel logit with random effects for blocker and rusher, weakly informative priors). They are **not** refitted here:

| Item | Value |
|---|---|
| Script | `scripts/confirmatory_analysis.py`, SHA-256 `8d842b70db03e2b0aa223f827bf0a496cb971ca3ea03f0a7c56563be8c046aca`, registered before any outcome was opened |
| Tag | `prereg-frozen` (pre-registration, script and its simulation tests committed together) |
| Command | `python scripts/confirmatory_analysis.py --mode confirmatory` (aborts unless script and input hashes and package versions match the manifest of pre-registration section 13.1) |
| Sampler | PyMC 6.4.0 with nutpie (numba backend, low-rank mass matrix), 4 cores per fit |
| Run | 2026-11-07, 64 minutes wall clock on 32 CPUs (single confirmatory run, no rerun) |
| Output | `reports/confirmatory_results.md` / `.json` (tables R0-R11), `reports/confirmatory/tables/*.csv` |

Table R2 (decisive checks) and the headline table R3 are read from `confirmatory_results.json`. Convergence criteria: R-hat <= 1.01, effective sample size >= 400, divergences <= 0.1%.

In [ ]:
with timed("6 frozen Bayes"):
    R = {k: pd.DataFrame(v) for k, v in FROZEN["tables"].items()}
    display(Markdown("**R2 - decisive checks and convergence (frozen run)**"))
    display(R["R2"])
    display(Markdown("**R3 - primary result, M1, registered composite, set P (frozen run)**"))
    display(R["R3"])
    display(Markdown("**Verdict sentence written by the frozen script**\n\n> " + FROZEN["verdict_sentence"]))

conv_rows = R["R2"][R["R2"].Check.str.startswith("Convergence")]
rhat = conv_rows.Observed.str.extract(r"max R-hat ([0-9.]+)")[0].astype(float)
ess = conv_rows.Observed.str.extract(r"min ESS bulk/tail (\d+)/(\d+)").astype(float)
div = conv_rows.Observed.str.extract(r"divergences (\d+)")[0].astype(int)
LABEL = "Opposite (qualified: Q1); Q4 failed (discordant: SA12, SA13)"
check("frozen run: mode confirmatory, status COMPLETE, no section 2.3 mismatches",
      FROZEN["mode"] == "confirmatory" and FROZEN["v0_struct_mismatches"] == [] and "COMPLETE" in (REPO / "reports" / "confirmatory_results.md").read_text(encoding="utf-8").split("## R1")[0])
check("frozen run: all 11 Bayesian fits converged (R-hat <= 1.01, ESS >= 400, no divergences)",
      len(conv_rows) == 11 and rhat.max() <= 1.01 and ess.min().min() >= 400 and div.sum() == 0, f"max R-hat {rhat.max()}, min ESS {int(ess.min().min())}")
check("frozen run: positive controls PC1 and PC2 and V0 passed", all(R["R2"].set_index("Check").loc[c, "Status"] == "PASS" for c in ("V0 counts and join", "PC1 pick coefficient", "PC1 dropback coefficient", "PC2 M2 d_lpick")))
check("frozen headline label", FROZEN["h1_final"]["label"] == LABEL, FROZEN["h1_final"]["label"])
check("frozen M1 gap: Bayes median -0.153, 95% interval [-0.299, -0.005]",
      (round(FROZEN["units"]["M1"]["logodds"]["median"], 3), round(FROZEN["decision"]["M1"]["lo"], 3), round(FROZEN["decision"]["M1"]["hi"], 3)) == (-0.153, -0.299, -0.005))
check("GLM mirror M1 agrees with the frozen GLM in the JSON (same sign, same estimate)", abs(fits["M1"].glm.term("gap")["est"] - FROZEN["glm_gap"]["M1"]["est"]) < 1e-6)
SCRIPT_SHA = sha256_file(SCRIPTS / "confirmatory_analysis.py")
print("frozen script sha256 in this checkout:", SCRIPT_SHA)

## 7. Figures

The figure scripts in `scripts/figures/` are run as they are (their output is redirected to the notebook output folder, and their data folder is pointed at the processed folder of this run). F4 and F6 read the frozen results: the notebook stages `confirmatory_results.json` and the small data products they need (`forest_data.csv`, `permutation_null.csv`, committed in `notebooks/frozen_products/`; table R4 is rebuilt from the JSON) in one folder.

* **F1** one engaged rep on the field (the movement itself). * **F2** the engagement measure picks out real contact. * **F3** how the Combine traits are measured from tracking. * **F5** anatomy of the surprise (own cell below). * **F4** forest plot: what the tracking gap adds over size, stopwatch, NGS score and draft slot. * **F6** robustness: permutation null, placebo and sensitivities.

In [ ]:
import shutil

STAGE = OUT_DIR / "frozen_results"
(STAGE / "tables").mkdir(parents=True, exist_ok=True)
shutil.copy(REPO / "reports" / "confirmatory_results.json", STAGE / "confirmatory_results.json")
for _k, _df in R.items():                                              # R1-R10 rebuilt from the JSON (the CSVs are git-ignored)
    _df.to_csv(STAGE / "tables" / f"{_k}.csv", index=False)
(STAGE / "reports" / "exploratory" / "tables").mkdir(parents=True, exist_ok=True)   # layout F5 expects under its ROOT
for f in ("x1_fits.csv", "x2_permutation_summary.csv", "x3_glm_fits.csv"):
    shutil.copy(REPO / "notebooks" / "frozen_products" / "exploratory_tables" / f, STAGE / "reports" / "exploratory" / "tables" / f)
for f in ("forest_data.csv", "permutation_null.csv"):
    shutil.copy(REPO / "notebooks" / "frozen_products" / f, STAGE / f)
    local = LOCAL_REFERENCE / "confirmatory" / f                       # the original run output, present on the author's machine only
    if local.exists():
        check(f"committed {f} is byte-identical to the frozen run output", sha256_file(local) == sha256_file(STAGE / f))

FIG_OUT = OUT_DIR / "figures"
FIG_OUT.mkdir(parents=True, exist_ok=True)


def run_figure(script, args=(), root=None):
    """Run a script of scripts/figures/ unchanged; only its data folder (PROC) and output folder (FIG) are redirected."""
    sdir = str(SCRIPTS / "figures")
    if sdir not in sys.path:
        sys.path.insert(0, sdir)
    import _common
    _common.PROC, _common.FIG = Path(paths.PROCESSED_DIR), FIG_OUT
    old_root = _common.ROOT
    if root is not None:
        _common.ROOT = Path(root)                              # F5 reads reports/exploratory/tables under ROOT
    old = sys.argv
    sys.argv = [script, *map(str, args)]
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            runpy.run_path(str(SCRIPTS / "figures" / script), run_name="__main__")
    finally:
        sys.argv = old
        _common.ROOT = old_root


FIGS = [("F1_engaged_rep", "f1_engaged_rep.py", ()), ("F2_engine_is_real", "f2_engine_is_real.py", (STAGE / "confirmatory_results.json",)),
        ("F3_combine_measured", "f3_combine_measured.py", ()), ("F4_forest", "f4_forest.py", (STAGE,)), ("F6_robustness", "f6_robustness.py", (STAGE,))]
for name, script, args in FIGS:
    with timed(f"7 figure {name}"):
        run_figure(script, args, root=STAGE if script == "f5_anatomy.py" else None)
    png = FIG_OUT / f"{name}.png"
    check(f"figure {name} written", png.exists() and png.stat().st_size > 20_000, f"{png.stat().st_size // 1024} KB")
    display(Markdown(f"**{name}**"))
    display(Image(filename=str(png), width=900))

### F5 - anatomy of the surprise

F5 puts the frozen blocker-side estimate (S1, and the pre-registered 5-frame sensitivity SA13) next to follow-up checks. **The exploratory rows come from `reports/specification_log.md` entries X1-X3, run after unsealing; they are not pre-registered, and the figure labels them EXPLORATORY.** Their result tables (`x1_fits.csv`, `x2_permutation_summary.csv`, `x3_glm_fits.csv`) are committed in `notebooks/frozen_products/exploratory_tables/` (the repository's `*.csv` ignore rule would otherwise hide them from the clone) and staged under the folder the script expects.

In [ ]:
with timed("7 figure F5_anatomy"):
    run_figure("f5_anatomy.py", (STAGE,), root=STAGE)
png = FIG_OUT / "F5_anatomy.png"
check("figure F5_anatomy written", png.exists() and png.stat().st_size > 20_000, f"{png.stat().st_size // 1024} KB")
display(Image(filename=str(png), width=900))

## 8. Reproducibility report

Package versions, the SHA-256 of the processed inputs and source files compared with the freeze manifest of pre-registration section 13.1, a content cross-check against the frozen local Parquet files (only on the author's machine, when this run used a different processed folder), a table of run times, and one PASS line per assertion of this notebook.

A hash mismatch of a *rebuilt* Parquet file is informational: the bytes depend on the writer's library version, whereas the numbers that matter are asserted above. Only on the author's machine, where the registered files themselves are reused, a mismatch fails the notebook.

In [ ]:
man = CA.parse_manifest((REPO / "reports" / "preregistration.md").read_text(encoding="utf-8"))
have = {"python": sys.version.split()[0]}
for p in ("polars", "pandas", "numpy", "scipy", "statsmodels", "matplotlib", "pyarrow", "duckdb"):
    try:
        have[p] = imd.version(p)
    except imd.PackageNotFoundError:
        have[p] = "not installed"
want = dict(x.strip().split(" ", 1) for x in man["PACKAGE_VERSIONS"].split(";") if x.strip())
display(pd.DataFrame([dict(package=k, registered_run=want.get(k, ""), this_environment=v, same=("yes" if want.get(k) == v else ("" if k not in want else "no"))) for k, v in have.items()]))

rows = []
for rel, registered in man["input_sha"].items():
    p = Path(paths.PROCESSED_DIR) / Path(rel).name if rel.startswith("data/processed/") else REPO / rel
    got = sha256_file(p) if p.exists() else "missing"
    rows.append(dict(file=rel, registered=registered[:12] + "...", this_run=got[:12] + "...", match="yes" if got == registered else "NO", reused=Path(paths.PROCESSED_DIR).resolve() == LOCAL_REFERENCE))
hash_tab = pd.DataFrame(rows)
rows.append(dict(file="scripts/confirmatory_analysis.py", registered=man["SCRIPT_SHA256"][:12] + "...", this_run=SCRIPT_SHA[:12] + "...", match="yes" if SCRIPT_SHA == man["SCRIPT_SHA256"] else "NO", reused=""))
display(pd.DataFrame(rows))
if Path(paths.PROCESSED_DIR).resolve() == LOCAL_REFERENCE:
    check("registered inputs (processed Parquet, traits.py, paths.py) match the freeze manifest hashes", (hash_tab.match == "yes").all())
else:
    print("processed folder was (re)built in this run: hash matches are informational,", int((hash_tab.match == "yes").sum()), "of", len(hash_tab), "identical")
print("frozen script hash matches the manifest:", SCRIPT_SHA == man["SCRIPT_SHA256"])

# content cross-check against the author's frozen Parquet files (when this run built its own)
if Path(paths.PROCESSED_DIR).resolve() != LOCAL_REFERENCE and LOCAL_REFERENCE.exists():
    with timed("8 content cross-check"):
        for t in list(paths.TABLES) + ["engaged_pairs", "player_baselines", "combine_traits", "engaged_outcomes_SEALED"]:
            a, b = Path(paths.PROCESSED_DIR) / f"{t}.parquet", LOCAL_REFERENCE / f"{t}.parquet"
            if a.exists() and b.exists():
                if t == "combine_traits":
                    pd.testing.assert_frame_equal(pd.read_parquet(a), pd.read_parquet(b), check_exact=False, rtol=1e-9, atol=1e-9, check_dtype=False)
                    same = True
                else:
                    same = pl.read_parquet(a).equals(pl.read_parquet(b))
                check(f"rebuilt {t}.parquet has the same content as the frozen local file", same)

display(Markdown("**Run time per step**"))
tt = pd.DataFrame([(k, round(v, 1)) for k, v in TIMES.items()], columns=["step", "seconds"])
display(tt)
print(f"total timed: {sum(TIMES.values()) / 60:.1f} min")

display(Markdown("**Assertions**"))
for name, ok, detail in CHECKS:
    print(("PASS  " if ok else "FAIL  ") + name)
print(f"\n{sum(ok for _, ok, _ in CHECKS)} of {len(CHECKS)} assertions passed")
assert all(ok for _, ok, _ in CHECKS)